# Phase 3 — Iterative Design Loop

This notebook demonstrates the core iterative design loop that drives
nanobody optimization. This is the heart of the NanoLFA-Design pipeline.

**The design loop:**
```
Predict (AF3) → Score → Filter → Diversify (ProteinMPNN) → Repeat
```

Each round takes the top candidates from the previous round, generates
CDR variants, predicts their complexes with the target, scores them,
and selects the best for the next round.

**What this notebook covers:**
1. Load seed scaffolds from Phase 2
2. Configure the iterative loop parameters
3. Walk through one round step-by-step (predict → score → filter → diversify)
4. Visualize scoring distributions and convergence
5. Set up W&B experiment tracking
6. Launch the full automated loop (local or HPC)

**Note:** Full AlphaFold and ProteinMPNN execution requires GPU access
and the respective tools installed. This notebook uses mock results
to demonstrate the pipeline logic when those tools are unavailable.

In [ ]:
import logging
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from nanolfa.core.pipeline import Candidate, RoundResult, NanoLFAPipeline
from nanolfa.core.config import load_config
from nanolfa.core.tracking import ExperimentTracker
from nanolfa.core.hpc import HPCManager, Scheduler
from nanolfa.scoring.composite import CompositeScorer
from nanolfa.filters.developability import DevelopabilityFilter

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

OUTPUT_DIR = Path('../data/results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Load Configuration and Seed Candidates

We load the PdG target config and the seed scaffolds from Phase 2.
The config defines all parameters for the design loop: number of rounds,
variants per round, scoring weights, decision thresholds, etc.

In [ ]:
# Load target config
config_path = '../configs/targets/pdg.yaml'

# For this notebook, load the config sections manually
# (full pipeline uses load_config which handles inheritance)
default_cfg = OmegaConf.load('../configs/default.yaml')
target_cfg = OmegaConf.load(config_path)
cfg = OmegaConf.merge(default_cfg, target_cfg)

print('Pipeline parameters:')
print(f'  Target: {cfg.target.name}')
print(f'  Max rounds: {cfg.pipeline.max_rounds}')
print(f'  Variants per round: {cfg.pipeline.variants_per_round}')
print(f'  Top-K advance: {cfg.pipeline.top_k_advance}')
print(f'  Convergence threshold: {cfg.pipeline.convergence_threshold}')
print(f'\nScoring weights:')
for metric, weight in cfg.scoring.weights.items():
    print(f'  {metric}: {weight}')

In [ ]:
# Load seed candidates from Phase 2
seed_fasta = Path('../data/templates/germline_vhh/scaffolds.fasta')

seeds = []
if seed_fasta.exists():
    from Bio import SeqIO
    for i, record in enumerate(SeqIO.parse(str(seed_fasta), 'fasta')):
        seeds.append(Candidate(
            candidate_id=record.id,
            sequence=str(record.seq),
            round_created=0,
        ))
    print(f'Loaded {len(seeds)} seed candidates from {seed_fasta}')
else:
    # Generate mock seeds for demonstration
    from nanolfa.utils.sequence import load_bundled_germlines
    scaffolds = load_bundled_germlines()
    for s in scaffolds:
        seeds.append(Candidate(
            candidate_id=s.name,
            sequence=s.sequence,
            round_created=0,
        ))
    print(f'Generated {len(seeds)} mock seed candidates from bundled germlines')

for s in seeds[:5]:
    print(f'  {s.candidate_id}: {len(s.sequence)} residues')

## 2. Understanding the Scoring Function

Before running the loop, let's understand how candidates are ranked.
The composite score combines six metrics with configurable weights.

We'll generate synthetic scores to demonstrate the scoring function
and decision thresholds.

In [ ]:
# Initialize the scorer
scorer = CompositeScorer(cfg.scoring)

# Generate synthetic score profiles to show how the composite works
np.random.seed(42)

profiles = {
    'Excellent binder': {
        'iptm': 0.88, 'plddt_interface': 89.0,
        'shape_complementarity': 0.74, 'binding_energy': -42.0,
        'buried_surface_area': 520.0,
    },
    'Good binder': {
        'iptm': 0.76, 'plddt_interface': 78.0,
        'shape_complementarity': 0.65, 'binding_energy': -32.0,
        'buried_surface_area': 450.0,
    },
    'Marginal binder': {
        'iptm': 0.63, 'plddt_interface': 68.0,
        'shape_complementarity': 0.57, 'binding_energy': -22.0,
        'buried_surface_area': 380.0,
    },
    'Non-binder': {
        'iptm': 0.45, 'plddt_interface': 55.0,
        'shape_complementarity': 0.42, 'binding_energy': -8.0,
        'buried_surface_area': 250.0,
    },
}

print(f'{"Profile":<20} {"Composite":>10} {"Tier":<10}')
print('-' * 45)
thresholds = cfg.scoring.thresholds
for name, raw in profiles.items():
    score = scorer.composite(raw, developability_score=0.8)
    if score >= thresholds.advance.composite_score_min:
        tier = 'GREEN'
    elif score >= thresholds.borderline.composite_score_min:
        tier = 'YELLOW'
    else:
        tier = 'RED'
    print(f'{name:<20} {score:>10.3f} {tier:<10}')

In [ ]:
# Visualize scoring weight contributions
fig, axes = plt.subplots(1, len(profiles), figsize=(16, 4), sharey=True)

metric_names = list(cfg.scoring.weights.keys())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#607D8B', '#E91E63']

for ax, (name, raw) in zip(axes, profiles.items()):
    # Compute per-metric weighted contributions
    contributions = []
    for metric in metric_names:
        if metric == 'developability':
            norm = 0.8  # mock developability
        else:
            norm = scorer._normalize(metric, raw.get(metric, 0))
        weight = cfg.scoring.weights[metric]
        contributions.append(norm * weight)

    ax.barh(metric_names, contributions, color=colors)
    ax.set_xlim(0, 0.3)
    ax.set_title(f'{name}\n(S={sum(contributions):.2f})', size=10)
    ax.set_xlabel('Weighted contribution')

axes[0].set_ylabel('Metric')
plt.suptitle('Scoring Function: Per-Metric Contributions', size=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scoring_contributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Simulated Design Loop

Since full AlphaFold / ProteinMPNN execution requires GPU infrastructure,
we simulate the design loop to demonstrate the pipeline logic, convergence
behavior, and visualization.

The simulation generates synthetic scores that improve over rounds
(mimicking successful optimization) with realistic noise.

In [ ]:
def simulate_round(
    candidates, round_num, n_variants=50, improvement_rate=0.03
):
    """Simulate one round of the design loop with synthetic scores."""
    rng = np.random.default_rng(42 + round_num)
    scored = []

    for cand in candidates:
        # Base score improves with rounds + random variation
        base = 0.45 + round_num * improvement_rate
        noise = rng.normal(0, 0.08)
        composite = np.clip(base + noise, 0, 1)

        # Individual metrics correlated with composite
        cand.iptm = np.clip(0.5 + composite * 0.4 + rng.normal(0, 0.05), 0.3, 0.95)
        cand.plddt_interface = np.clip(55 + composite * 35 + rng.normal(0, 5), 40, 95)
        cand.shape_complementarity = np.clip(0.4 + composite * 0.35 + rng.normal(0, 0.05), 0.3, 0.85)
        cand.binding_energy = -(10 + composite * 35 + rng.normal(0, 5))
        cand.buried_surface_area = 250 + composite * 400 + rng.normal(0, 50)
        cand.developability_score = np.clip(0.6 + rng.normal(0, 0.1), 0, 1)
        cand.composite_score = float(composite)

        # Classify tier
        if composite >= 0.70:
            cand.tier = 'green'
        elif composite >= 0.50:
            cand.tier = 'yellow'
        else:
            cand.tier = 'red'

        scored.append(cand)

    scored.sort(key=lambda c: c.composite_score, reverse=True)
    return scored


# Run simulated 5-round loop
n_rounds = 5
top_k = 50
round_results = []
current_pool = seeds.copy()

for round_num in range(1, n_rounds + 1):
    # Expand pool (simulate diversification)
    expanded = []
    for parent in current_pool:
        for v in range(6):  # 6 variants per parent
            variant = Candidate(
                candidate_id=f'r{round_num}_{parent.candidate_id}_v{v}',
                sequence=parent.sequence,  # mock — same seq
                round_created=round_num,
                parent_id=parent.candidate_id,
            )
            expanded.append(variant)

    scored = simulate_round(expanded, round_num)
    advanced = scored[:top_k]

    result = RoundResult(
        round_number=round_num,
        candidates_entered=len(expanded),
        candidates_predicted=len(expanded),
        candidates_passed_gates=len([c for c in scored if c.tier != 'red']),
        candidates_advanced=len(advanced),
        best_composite_score=advanced[0].composite_score,
        mean_composite_score=np.mean([c.composite_score for c in advanced]),
        median_iptm=np.median([c.iptm for c in advanced]),
        wall_time_hours=0.1 * round_num,
        advanced_candidates=advanced,
    )
    round_results.append(result)
    current_pool = advanced

    green = sum(1 for c in advanced if c.tier == 'green')
    yellow = sum(1 for c in advanced if c.tier == 'yellow')
    print(
        f'Round {round_num}: {len(expanded)} candidates -> '
        f'best={result.best_composite_score:.3f}, '
        f'mean={result.mean_composite_score:.3f}, '
        f'green={green}, yellow={yellow}'
    )

In [ ]:
# Convergence plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rounds = [r.round_number for r in round_results]
best_scores = [r.best_composite_score for r in round_results]
mean_scores = [r.mean_composite_score for r in round_results]
median_iptms = [r.median_iptm for r in round_results]

# Score convergence
ax = axes[0]
ax.plot(rounds, best_scores, 'o-', linewidth=2, markersize=8, label='Best', color='#E91E63')
ax.plot(rounds, mean_scores, 's--', linewidth=2, markersize=6, label='Mean (top-50)', color='#2196F3')
ax.axhline(y=0.70, color='green', linestyle=':', alpha=0.5, label='Green threshold')
ax.axhline(y=0.50, color='orange', linestyle=':', alpha=0.5, label='Yellow threshold')
ax.set_xlabel('Round', size=12)
ax.set_ylabel('Composite Score', size=12)
ax.set_title('Score Convergence', size=13)
ax.legend(fontsize=9)
ax.set_ylim(0.3, 0.9)

# Tier distribution per round
ax = axes[1]
greens = [sum(1 for c in r.advanced_candidates if c.tier == 'green') for r in round_results]
yellows = [sum(1 for c in r.advanced_candidates if c.tier == 'yellow') for r in round_results]
reds = [r.candidates_advanced - g - y for r, g, y in zip(round_results, greens, yellows)]
ax.bar(rounds, greens, color='#4CAF50', label='Green')
ax.bar(rounds, yellows, bottom=greens, color='#FF9800', label='Yellow')
ax.bar(rounds, reds, bottom=[g+y for g,y in zip(greens, yellows)], color='#F44336', label='Red')
ax.set_xlabel('Round', size=12)
ax.set_ylabel('Count', size=12)
ax.set_title('Tier Distribution', size=13)
ax.legend()

# Score delta (convergence indicator)
ax = axes[2]
deltas = [0] + [best_scores[i] - best_scores[i-1] for i in range(1, len(best_scores))]
colors = ['#4CAF50' if d > 0.02 else '#FF9800' if d > 0 else '#F44336' for d in deltas]
ax.bar(rounds, deltas, color=colors)
ax.axhline(y=0.02, color='black', linestyle='--', alpha=0.5, label='Convergence threshold')
ax.set_xlabel('Round', size=12)
ax.set_ylabel('Score Delta', size=12)
ax.set_title('Per-Round Improvement', size=13)
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'convergence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-metric distributions across rounds
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
metrics = [
    ('iptm', 'ipTM'),
    ('plddt_interface', 'Interface pLDDT'),
    ('shape_complementarity', 'Shape Complementarity'),
    ('binding_energy', 'Binding Energy (REU)'),
    ('buried_surface_area', 'BSA (A^2)'),
    ('composite_score', 'Composite Score'),
]

for ax, (attr, label) in zip(axes.flat, metrics):
    data_per_round = []
    for r in round_results:
        values = [getattr(c, attr) for c in r.advanced_candidates if getattr(c, attr) is not None]
        data_per_round.append(values)

    bp = ax.boxplot(data_per_round, labels=[str(r.round_number) for r in round_results],
                    patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Blues(np.linspace(0.3, 0.8, n_rounds))):
        patch.set_facecolor(color)
    ax.set_xlabel('Round')
    ax.set_ylabel(label)
    ax.set_title(label, size=11)

plt.suptitle('Metric Distributions Across Design Rounds', size=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Experiment Tracking with Weights & Biases

The ExperimentTracker logs all metrics, candidate tables, and artifacts
to W&B for live monitoring during HPC runs.

To enable:
1. `pip install wandb`
2. `wandb login` (one-time API key setup)
3. Set `experiment_tracker: wandb` in config

In [ ]:
# Initialize tracker (degrades gracefully if wandb not installed)
tracker = ExperimentTracker(cfg)

# Log the simulated round results
tracker.init_run(target='pdg', tags=['demo', 'simulated'])

for result in round_results:
    tracker.log_round(result)
    tracker.log_candidates(result.advanced_candidates, result.round_number)

tracker.finish()
print('Tracking complete (check W&B dashboard if enabled).')

## 5. HPC Job Submission

For real pipeline runs on a Slurm or PBS cluster, the HPCManager handles
job submission, dependency chaining, and status monitoring.

The pipeline can either:
- Run as a single long job (`configs/hpc/slurm_full_pipeline.sh`)
- Submit each round as a separate job with dependencies

In [ ]:
# Show what a Slurm submission would look like
print('Example: submitting individual rounds with dependencies')
print('=' * 60)
print()

for round_num in range(1, 6):
    deps = f'--dependency=afterok:$JOB_{round_num - 1}' if round_num > 1 else ''
    print(f'# Round {round_num}')
    print(f'JOB_{round_num}=$(sbatch {deps} \\') 
    print(f'  --export=TARGET=pdg,ROUND={round_num},N_VARIANTS=300 \\')
    print(f'  configs/hpc/slurm_af_round.sh | awk \'{{print $4}}\')')
    print(f'echo "Round {round_num}: job $JOB_{round_num}"')
    print()

In [ ]:
# The HPCManager can also be used programmatically:
print('Programmatic usage (local mode for demo):')
print()

hpc = HPCManager(cfg.compute)
print(f'Scheduler: {hpc.scheduler.value}')
print(f'Partition: {hpc.partition}')
print(f'GPUs: {hpc.gpus_per_node}')
print(f'Memory: {hpc.memory_gb} GB')
print(f'Time limit: {hpc.time_limit_hours} hours')
print()
print('To submit a real job:')
print('  job = hpc.submit(')
print('      command=["python", "scripts/run_design_round.py", ...],')
print('      job_name="af3_round_01",')
print('      work_dir=Path("data/results/round_01"),')
print('  )')
print('  hpc.wait(job)  # blocks until complete')

## 6. Running the Real Pipeline

When running on actual infrastructure with AlphaFold 3 and ProteinMPNN
installed, the full pipeline is launched with:

```bash
# Single-GPU local run
python scripts/run_pipeline.py \
    --config configs/targets/pdg.yaml \
    --rounds 5 \
    --seed-fasta data/templates/germline_vhh/scaffolds.fasta

# HPC submission (Slurm)
sbatch --export=TARGET=pdg,ROUNDS=5 configs/hpc/slurm_full_pipeline.sh

# Or via Makefile
make run-pdg
```

The pipeline will:
1. Load seeds and target config
2. Run 5 rounds of predict → score → filter → diversify
3. Log all metrics to W&B (if enabled)
4. Save per-round FASTA + TSV to `data/results/round_XX/`
5. Stop early if convergence is reached (score delta < 0.02)
6. Output final top-20 candidates for experimental validation

---

**Next:** Phase 4 — Specificity Engineering (notebook `04_specificity_analysis.ipynb`)